In [1]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

!pip install -U bitsandbytes
!pip install accelerate transformers


In [4]:
# ✅ Base chat model
TUNE_MODEL = "./mistral7b_subject_merged"
DATA_PATH = "fine_tune_data.jsonl"

In [5]:
# ---- Load dataset ----
prompts, completions = [], []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()

        # Skip empty line
        if not line:
            continue

        # Remove BOM
        if line.startswith("\ufeff"):
            line = line.replace("\ufeff", "")

        try:
            d = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"❌ JSON error in line {i}: {e}")
            print("Line content:", repr(line))
            continue 

        prompts.append(d["prompt"])
        completions.append(d["completion"])

print(f"Loaded {len(prompts)} samples")

Loaded 7475 samples


In [6]:
# ---- Load model ----
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # A10G supports bf16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(TUNE_MODEL, use_fast=True)

# Leave headroom so auto-sharding never tries CPU/disk
max_mem = {0: "22GiB", "cpu": "0GiB"}  # restrict CPU offload explicitly

model = AutoModelForCausalLM.from_pretrained(
    TUNE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",        
    max_memory=max_mem,
    low_cpu_mem_usage=True,
)

model.eval()
print("Loaded on:", next(model.parameters()).device)

2025-11-19 19:20:39.712907: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763580039.737189   20117 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763580039.744746   20117 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-19 19:20:39.796723: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded on: cuda:0


In [7]:
# ---- Chat prompt format for BioMistral ----
def build_prompt(symptoms):
    return f"""
### Instruction:
You are a medical diagnosis assistant. Based on the given patient symptoms, output the most likely disease and the appropriate medical department.

Return answer STRICTLY in this format:

Disease: <disease name>
Department: <department name>

If unsure, output the most reasonable guess based on common medical knowledge.

Symptoms: {symptoms}

### Response:
""".strip()


In [8]:
import re

def format_output(text):
    # normalize lowercase
    text = text.strip()

    disease = "Unknown"
    department = "Unknown"

    # extract disease
    m = re.search(r"Disease\s*:\s*(.+)", text, re.IGNORECASE)
    if m:
        disease = m.group(1).strip()

    # extract department
    m = re.search(r"Department\s*:\s*(.+)", text, re.IGNORECASE)
    if m:
        department = m.group(1).strip()

    return f"Disease: {disease}\nDepartment: {department}"

In [10]:
def generate_tune(symptoms):
    prompt = build_prompt(symptoms)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    text = tokenizer.decode(output[0], skip_special_tokens=True)

    if text.startswith(prompt):
        text = text[len(prompt):]

    return format_output(text)

In [11]:
# ---- Run tuned model on first N samples ----
N = 50
tune_outputs = [generate_tune(p) for p in prompts[:N]]


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

In [12]:
# ---- Save result ----
df = pd.DataFrame({
    "symptom_prompt": prompts[:N],
    "tune_output": tune_outputs,
    "expected_completion": completions[:N]
})

df.to_csv("project1_tune_results_biomistral.csv", index=False)
df.head()


,symptom_prompt,tune_output,expected_completion
0,"Given the following symptoms, identify the mos...",Disease: allergy\nDepartment: dermatology / al...,Disease: allergy\nDepartment: dermatology /
1,"Given the following symptoms, identify the mos...",Disease: allergy\nDepartment: dermatology / al...,Disease: allergy\nDepartment: dermatology /
2,"Given the following symptoms, identify the mos...",Disease: allergy\nDepartment: dermatology / al...,Disease: allergy\nDepartment: dermatology /
3,"Given the following symptoms, identify the mos...",Disease: allergy\nDepartment: dermatology / al...,Disease: allergy\nDepartment: dermatology /
4,"Given the following symptoms, identify the mos...",Disease: allergy\nDepartment: dermatology / al...,Disease: allergy\nDepartment: dermatology /


In [13]:
import pandas as pd
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction


nltk.download('punkt')
nltk.download('punkt_tab')

df = pd.read_csv("project1_tune_results_biomistral.csv")

smooth = SmoothingFunction().method3
bleu_scores = []

for ref, pred in zip(df["expected_completion"], df["tune_output"]):
    ref_tokens = nltk.word_tokenize(str(ref))
    pred_tokens = nltk.word_tokenize(str(pred))

    score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
    bleu_scores.append(score)

df["BLEU"] = bleu_scores

print(f"Average BLEU: {df['BLEU'].mean():.4f}")


#df.to_csv("tune_results_with_BLEU.csv", index=False)
#print("Saved to tune_results_with_BLEU.csv")

#df.head()

Average BLEU: 0.7216


[nltk_data] Downloading package punkt to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
